# 04. 시간 경과와 날짜별 이력

- **theta**: 시장 그대로 평가일만 이동 (`Market.roll`, ForwardForward: 오늘의 선도가 실현).
- **이력**: 날짜마다 시장을 만들고 **결과 숫자만 표로** 남긴다.

> 데이터: `tests/fixtures/market_sample.txt`는 **합성 샘플**(`날짜 키 값`)이다. 흐름을 익히는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.market.quotes import read_quotes

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
cfg = load_config(ROOT / "configs")
quotes = read_quotes(ROOT / "tests" / "fixtures" / "market_sample.txt")  # 합성 샘플
ASOF = date(2026, 10, 23)

from quanttool.analysis.risk import theta
from quanttool.market.curves import curve_shape, curve_table
from quanttool.portfolio import Portfolio

market = build_market(ASOF, quotes[ASOF], cfg)
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml").build(market, cfg)

## 1. theta

In [ ]:
theta(book, market, date(2026, 10, 26)).round(2)  # 다음 영업일 (주말 포함 3일)

## 2. 날짜별 커브 이력

In [ ]:
zero_hist, shape_hist = {}, {}
for d, q in quotes.items():
    m = build_market(d, q, cfg)
    with m.active():
        table = curve_table(m["UST"], cfg.conventions["UST"])
    zero_hist[d] = table["zero_cc"] * 100
    shape_hist[d] = curve_shape(table)
display(pd.DataFrame(zero_hist).T.round(4))
pd.DataFrame(shape_hist).T

## 3. 평가일이 바꾸는 것
커브는 asof에 고정이고 NPV도 커브 기준일로 할인하지만, 결제일·경과이자·clean 가격은 평가일에서 정해진다. 그래서 상품 값은 그 시장의 `active()` 안에서 본다 (`Portfolio.npvs`는 알아서 그렇게 한다).

In [ ]:
from quanttool.market.curves import evaluation_date

bond = book.instruments["UST_10Y"]
rows = []
for label, ctx in [
    ("2026-10-23 (active)", market.active()),
    ("2027-02-16", evaluation_date(date(2027, 2, 16))),
]:
    with ctx:
        rows.append(
            {
                "평가일": label,
                "결제일": bond.settlementDate().ISO(),
                "경과이자": bond.accruedAmount(),
                "clean": bond.cleanPrice(),
                "NPV": bond.NPV(),
            }
        )
pd.DataFrame(rows).set_index("평가일").round(4)